# Financial Risk Intelligence Platform - EDA (Phase 1)

This notebook focuses on:

- loading raw transaction data
- checking data quality (nulls, duplicates, dtypes)
- understanding fraud class imbalance
- exploring `Amount` and `Time` behavior
- documenting key insights and limitations

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.load_data import load_raw_data

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 120)

## Load Data

In [ ]:
try:
    df = load_raw_data()
except FileNotFoundError as error:
    raise FileNotFoundError(
        f"{error}\n\nPlace a CSV file in data/raw/ (e.g., transactions.csv, creditcard.csv, or train_transaction.csv)."
    )

df.shape

In [ ]:
df.head()

## Data Structure and Types

In [ ]:
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")

df.info()

## Missing Values and Duplicates

In [ ]:
missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame('missing_count')
)
missing_summary['missing_pct'] = (missing_summary['missing_count'] / len(df) * 100).round(2)

missing_summary.head(20)

In [ ]:
duplicate_count = int(df.duplicated().sum())
print(f"Duplicate rows: {duplicate_count}")

## Target Distribution

In [ ]:
target_candidates = ['isFraud', 'Class', 'fraud', 'target']
target_col = next((col for col in target_candidates if col in df.columns), None)

if target_col is None:
    raise ValueError(f"No target column found. Expected one of: {target_candidates}")

target_counts = df[target_col].value_counts(dropna=False).sort_index()
target_pct = (target_counts / len(df) * 100).round(2)

summary = pd.DataFrame({
    'count': target_counts,
    'percentage': target_pct,
})
summary

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target_col)
plt.title(f'Target Distribution ({target_col})')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
fraud_rate = df[target_col].mean() if set(df[target_col].dropna().unique()).issubset({0, 1}) else None

if fraud_rate is not None:
    print(f"Fraud percentage: {fraud_rate * 100:.2f}%")
    print(f"Non-fraud percentage: {(1 - fraud_rate) * 100:.2f}%")
else:
    print('Target is not strictly binary {0,1}. Review label encoding first.')

## Amount Analysis

In [ ]:
amount_col = 'Amount' if 'Amount' in df.columns else ('TransactionAmt' if 'TransactionAmt' in df.columns else None)

if amount_col is None:
    print('No amount column found (Amount/TransactionAmt).')
else:
    print(df[amount_col].describe())

In [ ]:
if amount_col is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.histplot(df[amount_col], bins=50, ax=axes[0], kde=True)
    axes[0].set_title(f'{amount_col} Distribution')

    sns.boxplot(x=df[amount_col], ax=axes[1])
    axes[1].set_title(f'{amount_col} Boxplot')

    plt.tight_layout()
    plt.show()

## Time Analysis

In [ ]:
time_col = 'Time' if 'Time' in df.columns else ('TransactionDT' if 'TransactionDT' in df.columns else None)

if time_col is None:
    print('No time column found (Time/TransactionDT).')
else:
    print(df[time_col].describe())

In [ ]:
if time_col is not None:
    plt.figure(figsize=(10, 4))
    sns.histplot(df[time_col], bins=60)
    plt.title(f'{time_col} Distribution')
    plt.tight_layout()
    plt.show()

## Correlations (Numeric Features vs Target)

In [ ]:
numeric_df = df.select_dtypes(include=['number']).copy()

if target_col in numeric_df.columns:
    corr_with_target = (
        numeric_df.corr(numeric_only=True)[target_col]
        .drop(labels=[target_col], errors='ignore')
        .sort_values(key=lambda s: s.abs(), ascending=False)
    )
    display(corr_with_target.head(15).to_frame('correlation'))
else:
    print('Target is not numeric. Correlation with target skipped.')

In [ ]:
if target_col in numeric_df.columns:
    top_features = corr_with_target.head(10).index.tolist()
    cols_for_heatmap = [target_col] + top_features
    corr_matrix = numeric_df[cols_for_heatmap].corr(numeric_only=True)

    plt.figure(figsize=(9, 6))
    sns.heatmap(corr_matrix, cmap='coolwarm', center=0)
    plt.title('Top Numeric Correlations with Target')
    plt.tight_layout()
    plt.show()

## Key Insights

- The dataset requires class-imbalance-aware evaluation due to low fraud prevalence.
- Amount-related variables are typically right-skewed and may benefit from log scaling in Phase 2.
- Duplicate and missing-value handling should be standardized before feature engineering.
- Time-like variables can capture temporal patterns and transaction velocity behavior.
- Correlation is useful for quick screening, but non-linear fraud patterns will still require robust models.

## Limitations

- Public fraud datasets may not represent current production fraud behavior.
- Many variables are anonymized, limiting business interpretability.
- Correlation analysis is linear and does not capture complex interactions.
- Label quality and potential data drift are not validated in this phase.
- This notebook is exploratory and should be complemented with reproducible scripts in `src/data/`.